# MISION: Preparar el Quirofano

**Problema:** Datos mixtos — numericos + categoricos + nulos. El modelo no entiende "Male" ni "Private". Necesitamos traducirlo todo a numeros.

---

### Retos
1. **Nulos:** BMI tiene 201 valores faltantes
2. **Categoricas:** 5 columnas con texto que los algoritmos no digieren
3. **Desbalance extremo:** 95.1% vs 4.9% — el split debe ser estratificado
4. **Escalas distintas:** edad (0-82), glucosa (55-272), BMI (10-97) — necesitan normalizacion

**Objetivo:** Pasar de 11 columnas crudas a una matriz de features limpia, codificada y escalada, lista para alimentar modelos.

## 1. Imports y Configuracion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings
warnings.filterwarnings('ignore')

# Estilo y colores
plt.style.use('seaborn-v0_8-whitegrid')
NO_STROKE = '#2ecc71'
STROKE = '#e74c3c'

# Rutas
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
data_dir = os.path.join(project_root, "data", "heart_attack")
processed_dir = os.path.join(data_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

print(f"Datos: {data_dir}")
print(f"Salida: {processed_dir}")

## 2. Carga de Datos (autocontenido)

In [ ]:
df = pd.read_csv(os.path.join(data_dir, "healthcare-dataset-stroke-data.csv"))
df.drop(columns=['id'], inplace=True)
print(f"Dimensiones originales: {df.shape}")
print(f"Columnas: {list(df.columns)}")
print(f"\nNulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Imputacion de BMI por Grupo de Edad

BMI varia con la edad. Imputar con la mediana global seria burdo. Mejor: mediana por grupo etario.

In [ ]:
# Definir bins de edad
age_bins = [0, 18, 40, 60, 100]
age_labels = ['0-18', '18-40', '40-60', '60+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=False)

# Mediana de BMI por grupo
bmi_medians = df.groupby('age_group')['bmi'].median()
print("Mediana de BMI por grupo de edad:")
print(bmi_medians)
print(f"\nNulos en BMI antes: {df['bmi'].isnull().sum()}")

In [ ]:
# Guardar distribucion original para comparar
bmi_before = df['bmi'].copy()

# Imputar
for group in age_labels:
    mask = (df['age_group'] == group) & (df['bmi'].isnull())
    df.loc[mask, 'bmi'] = bmi_medians[group]

print(f"Nulos en BMI despues: {df['bmi'].isnull().sum()}")

In [ ]:
# Comparar distribuciones antes/despues
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bmi_before.dropna(), bins=30, color=NO_STROKE, alpha=0.7, edgecolor='white')
axes[0].set_title('BMI - Antes (sin nulos)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('BMI')
axes[0].axvline(bmi_before.median(), color=STROKE, linestyle='--', label=f'Mediana: {bmi_before.median():.1f}')
axes[0].legend()

axes[1].hist(df['bmi'], bins=30, color=NO_STROKE, alpha=0.7, edgecolor='white')
axes[1].set_title('BMI - Despues (imputado por grupo)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('BMI')
axes[1].axvline(df['bmi'].median(), color=STROKE, linestyle='--', label=f'Mediana: {df["bmi"].median():.1f}')
axes[1].legend()

plt.suptitle('Distribucion de BMI: Antes vs Despues de Imputacion', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("La distribucion se mantiene practicamente igual. Imputacion conservadora.")

In [ ]:
# Eliminar columna auxiliar
df.drop(columns=['age_group'], inplace=True)

## 4. Smoking Status: "Unknown" se queda

"Unknown" no es un error — puede ser que no se pregunto, lo cual correlaciona con un perfil demografico distinto. Lo tratamos como categoria propia.

In [ ]:
print("Distribucion de smoking_status:")
print(df['smoking_status'].value_counts())
print(f"\nTasa de stroke por smoking_status:")
print((df.groupby('smoking_status')['stroke'].mean() * 100).round(2).to_string())
print("\n'Unknown' tiene tasa de stroke distinta a las demas. Confirma que es informativo.")

## 5. Filtrar gender="Other"

Solo 1 caso. No aporta informacion estadistica y puede causar ruido.

In [ ]:
print(f"Registros con gender='Other': {(df['gender'] == 'Other').sum()}")
df = df[df['gender'] != 'Other'].reset_index(drop=True)
print(f"Dimensiones tras filtro: {df.shape}")

## 6. Codificacion de Variables Categoricas

Dos estrategias:
- **Binarias:** mapeo directo a 0/1
- **Multiclase:** One-Hot Encoding (drop_first para evitar colinealidad)

In [ ]:
# Codificacion binaria
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
df['ever_married'] = df['ever_married'].map({'Yes': 1, 'No': 0})
df['Residence_type'] = df['Residence_type'].map({'Urban': 1, 'Rural': 0})

print("Codificacion binaria aplicada:")
print(f"  gender:         Male=1, Female=0")
print(f"  ever_married:   Yes=1, No=0")
print(f"  Residence_type: Urban=1, Rural=0")
print(f"\nVerificacion:")
print(df[['gender', 'ever_married', 'Residence_type']].describe())

In [ ]:
# One-Hot Encoding para work_type y smoking_status
print(f"Categorias en work_type: {df['work_type'].unique()}")
print(f"Categorias en smoking_status: {df['smoking_status'].unique()}")

df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first=True, dtype=int)

print(f"\nColumnas tras One-Hot Encoding:")
print(list(df.columns))
print(f"\nDimensiones: {df.shape}")

## 7. Separacion de Features y Target

In [ ]:
X = df.drop(columns=['stroke'])
y = df['stroke']

print(f"Features (X): {X.shape}")
print(f"Target (y):   {y.shape}")
print(f"\nColumnas de X:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col} ({X[col].dtype})")

## 8. Train/Test Split (Estratificado)

Con solo 4.9% de positivos, un split aleatorio podria dejar el test set sin casos de stroke. **Stratify** garantiza la misma proporcion en ambos sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nProporcion de stroke:")
print(f"  Train: {y_train.mean()*100:.2f}%")
print(f"  Test:  {y_test.mean()*100:.2f}%")
print(f"  Total: {y.mean()*100:.2f}%")
print("\nProporciones practicamente identicas. Stratify funciona.")

## 9. Escalado con StandardScaler

Solo las columnas numericas continuas necesitan escalado. Las binarias y one-hot ya estan en rango adecuado.

**Regla de oro:** fit en train, transform en train Y test. Nunca fit en test.

In [ ]:
numeric_to_scale = ['age', 'avg_glucose_level', 'bmi']

scaler = StandardScaler()

# Fit SOLO en train
X_train[numeric_to_scale] = scaler.fit_transform(X_train[numeric_to_scale])
# Transform test con los parametros de train
X_test[numeric_to_scale] = scaler.transform(X_test[numeric_to_scale])

print("Parametros del scaler (aprendidos del train):")
for col, mean, std in zip(numeric_to_scale, scaler.mean_, scaler.scale_):
    print(f"  {col:<20} media={mean:.2f}  std={std:.2f}")

print(f"\nVerificacion en train (deben ser ~0 y ~1):")
print(X_train[numeric_to_scale].describe().loc[['mean', 'std']].round(4))

## 10. Matriz Final de Features

In [ ]:
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nColumnas ({X_train.shape[1]} features):")
for i, col in enumerate(X_train.columns, 1):
    print(f"  {i:2d}. {col:<35} dtype={X_train[col].dtype}")

print(f"\nPrimeras 5 filas de X_train:")
X_train.head()

In [ ]:
X_train.describe().round(3)

## 11. Guardar en Parquet

In [ ]:
X_train.to_parquet(os.path.join(processed_dir, 'X_train.parquet'), index=False)
X_test.to_parquet(os.path.join(processed_dir, 'X_test.parquet'), index=False)
y_train.to_frame().to_parquet(os.path.join(processed_dir, 'y_train.parquet'), index=False)
y_test.to_frame().to_parquet(os.path.join(processed_dir, 'y_test.parquet'), index=False)

print("Archivos guardados en:")
for f in ['X_train.parquet', 'X_test.parquet', 'y_train.parquet', 'y_test.parquet']:
    fpath = os.path.join(processed_dir, f)
    size = os.path.getsize(fpath) / 1024
    print(f"  {f:<20} ({size:.1f} KB)")

## 12. Resumen del Pipeline

| Paso | Accion | Resultado |
|------|--------|-----------|
| 1 | Drop `id` | 12 -> 11 columnas |
| 2 | Imputar BMI por grupo de edad | 201 nulos eliminados |
| 3 | Mantener smoking_status "Unknown" | Categoria informativa |
| 4 | Filtrar gender="Other" | 5,110 -> 5,109 filas |
| 5 | Codificacion binaria | gender, ever_married, Residence_type -> 0/1 |
| 6 | One-Hot Encoding | work_type (4 dummies), smoking_status (3 dummies) |
| 7 | Train/Test split estratificado | 80/20, misma proporcion de stroke |
| 8 | StandardScaler | age, avg_glucose_level, bmi normalizados |
| 9 | Guardar en Parquet | 4 archivos listos para modelado |

**Resultado:** De 11 features originales a ~15 features codificadas, limpias y escaladas.

**Siguiente paso:** Modelado con manejo de desbalance (SMOTE, class_weight, threshold tuning).